# 01 — Auditoria do HomeObjects-3K

Este notebook realiza a primeira etapa prática do TCC: verificar a estrutura, a qualidade e a distribuição do dataset antes do treinamento.

**Por que isso é necessário?** Um modelo pode apresentar resultados enganosos quando existem classes desbalanceadas, imagens duplicadas entre os conjuntos ou anotações incorretas. A auditoria transforma essas verificações em evidências que poderão ser descritas no Capítulo 4.

## 1. Importação das bibliotecas e localização do projeto

O código abaixo funciona mesmo que o notebook seja aberto a partir da pasta `notebooks`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import torch
import yaml
from IPython.display import Image as ImagemNotebook
from IPython.display import display

def localizar_raiz_projeto() -> Path:
    pasta_atual = Path.cwd().resolve()
    for candidata in [pasta_atual, *pasta_atual.parents]:
        if (candidata / 'requirements.txt').exists() and (candidata / 'src').exists():
            return candidata
    raise FileNotFoundError('A raiz do projeto não foi encontrada.')

RAIZ_PROJETO = localizar_raiz_projeto()
if str(RAIZ_PROJETO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROJETO))

from src.auditoria_dados import auditar_dataset, baixar_homeobjects

print(f'Raiz do projeto: {RAIZ_PROJETO}')
print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponível: {torch.cuda.is_available()}')

## 2. Leitura das configurações

O arquivo YAML centraliza o endereço do dataset, a semente e os nomes das classes. Dessa forma, as decisões não ficam escondidas dentro do notebook.

In [ ]:
caminho_config = RAIZ_PROJETO / 'configs' / 'projeto.yaml'

with caminho_config.open(encoding='utf-8') as arquivo:
    configuracao = yaml.safe_load(arquivo)

CLASSES = {int(chave): valor for chave, valor in configuracao['classes'].items()}
URL_DATASET = configuracao['dataset']['url']
PASTA_RAW = RAIZ_PROJETO / configuracao['dataset']['pasta_raw']
PASTA_RELATORIOS = RAIZ_PROJETO / 'reports'

print(f'Dataset: {configuracao["dataset"]["nome"]}')
print(f'Quantidade de classes previstas: {len(CLASSES)}')
display(pd.DataFrame(CLASSES.items(), columns=['id_classe', 'classe']))

## 3. Download e extração

O download só ocorre quando o dataset ainda não existe em `data/raw`. O arquivo original não será alterado durante as próximas etapas.

In [ ]:
raiz_dataset = baixar_homeobjects(
    pasta_raw=PASTA_RAW,
    url=URL_DATASET,
)

print(f'Raiz identificada: {raiz_dataset}')
print('Pastas encontradas:')
for caminho in sorted(raiz_dataset.iterdir()):
    print(f'  - {caminho.name}')

## 4. Execução da auditoria

Nesta etapa serão verificadas imagens, anotações YOLO, distribuição das classes, dimensões das caixas e duplicidades. Dependendo do computador, o cálculo dos hashes pode levar alguns minutos.

In [ ]:
resultados = auditar_dataset(
    raiz_dataset=raiz_dataset,
    pasta_relatorios=PASTA_RELATORIOS,
    classes=CLASSES,
)

print('Auditoria concluída.')

## 5. Quantidade de imagens por conjunto

O HomeObjects-3K possui originalmente treino e validação, mas não possui teste independente. Mais adiante, parte do treino será usada como validação de desenvolvimento e o conjunto oficial de validação será preservado para o teste interno final.

In [ ]:
display(resultados['resumo'])

## 6. Distribuição das classes

É importante observar tanto o número de instâncias quanto o número de imagens. Muitas caixas de uma classe concentradas em poucas imagens não representam a mesma diversidade de uma classe distribuída em vários ambientes.

In [ ]:
df_classes = resultados['classes'].sort_values('instancias', ascending=False)
display(df_classes)

instancias_positivas = df_classes.loc[df_classes['instancias'] > 0, 'instancias']
if not instancias_positivas.empty:
    razao_desbalanceamento = instancias_positivas.max() / instancias_positivas.min()
    print(f'Razão entre a maior e a menor classe: {razao_desbalanceamento:.2f}x')

In [ ]:
caminho_grafico_classes = PASTA_RELATORIOS / 'figures' / 'distribuicao_instancias.png'
display(ImagemNotebook(filename=str(caminho_grafico_classes)))

## 7. Problemas encontrados

Arquivos de anotação ausentes ou vazios não são automaticamente erros: podem representar imagens negativas. Esses casos devem ser revisados visualmente antes de qualquer correção.

In [ ]:
df_problemas = resultados['problemas']

if df_problemas.empty:
    print('Nenhum problema estrutural foi encontrado.')
else:
    display(
        df_problemas.groupby('tipo')
        .size()
        .rename('quantidade')
        .sort_values(ascending=False)
        .reset_index()
    )
    display(df_problemas.head(20))

## 8. Duplicidades

Duplicidades entre treino e validação podem aumentar artificialmente as métricas. O SHA-256 identifica arquivos idênticos; o dHash igual ajuda a encontrar imagens visualmente semelhantes mesmo quando o arquivo foi recodificado.

In [ ]:
df_duplicidades = resultados['duplicidades']

if df_duplicidades.empty:
    print('Nenhuma duplicidade foi encontrada pelos critérios iniciais.')
else:
    print(f'Grupos de duplicidades: {len(df_duplicidades)}')
    display(df_duplicidades)

## 9. Inspeção visual das anotações

A grade abaixo apresenta pelo menos uma imagem de cada classe encontrada. Caixas vermelhas indicam a classe destacada na célula; caixas verdes representam outros objetos anotados na mesma imagem.

In [ ]:
caminho_amostras = PASTA_RELATORIOS / 'figures' / 'amostras_anotadas.jpg'
if caminho_amostras.exists():
    display(ImagemNotebook(filename=str(caminho_amostras)))
else:
    print('A grade não foi gerada porque nenhuma caixa válida foi encontrada.')

## 10. Checklist para decisão do grupo

Após revisar as tabelas e imagens, o grupo deve registrar as respostas abaixo antes de iniciar o treinamento:

- [ ] Existem classes com poucas imagens ou instâncias?
- [ ] Existem caixas claramente incorretas ou objetos sem anotação?
- [ ] Há imagens repetidas entre treino e validação?
- [ ] As classes `tv`, `laptop`, `window` e `photo frame` serão mantidas?
- [ ] A classe `lamp` representa luminárias de piso ou principalmente luminárias de mesa?
- [ ] `door` será tratada como obstáculo ou como elemento de passagem?
- [ ] As classes críticas definidas em `configs/projeto.yaml` continuam adequadas?

**Próxima etapa:** com essas respostas, criar uma taxonomia definitiva, congelar os conjuntos e desenvolver o treinamento baseline.